In [6]:
import os 
import pandas as pd

cwd = os.getcwd()
sep  = os.sep

path = cwd + sep + "experimental_pdbs" + sep + "no_complex"
csv_file = cwd + sep + "experimental_pdbs.csv"
#Read csv file
df = pd.read_csv(csv_file)
df

,Mutation,PDB ID
0,A109S,NO
1,A109V,5FW7
2,A120S,8W1N
3,A19D,5DEJ
4,A25S,NO
...,...,...
128,Y114S,NO
129,Y116S,NO
130,Y69H,NO
131,Y69I,NO


In [7]:
#Count mutations with NO experimental PDBs: "Mutation" column is 'NO'
count_no_pdbs = df[df['PDB ID'] == 'NO'].shape[0]
print(f"Number of mutations with NO experimental PDBs: {count_no_pdbs}")

Number of mutations with NO experimental PDBs: 111


In [8]:
count_pdbs = df[df['PDB ID'] != 'NO'].shape[0]
print(f"Number of mutations with experimental PDBs: {count_pdbs}")

Number of mutations with experimental PDBs: 22


In [9]:
# Drop rows where 'PDB ID' is 'NO'
df = df[df['PDB ID'] != 'NO'] 
df

,Mutation,PDB ID
1,A109V,5FW7
2,A120S,8W1N
3,A19D,5DEJ
5,A25T,3TFB
28,E54G,3A4E
29,E54K,3A4F
33,E61K,6KGB
36,E89K,5FW8
71,I84S,2NOY
78,L55P,4TKW


In [10]:
import urllib

def get_pdb(pdb_id):
    """Get the PDB file from rcsb."""
    urllib.request.urlretrieve(f'http://files.rcsb.org/download/{pdb_id}.pdb', f'{pdb_id}.pdb')

In [11]:
#Get pdbs from RCSB
for index, row in df.iterrows():
    pdb_id = row['PDB ID']
    if not os.path.exists(f"{path}{sep}{pdb_id}.pdb"):
        print(f"Downloading {pdb_id}.pdb")
        get_pdb(pdb_id)
        os.rename(f"{pdb_id}.pdb", f"{path}{sep}{pdb_id}.pdb")
    else:
        print(f"{pdb_id}.pdb already exists, skipping download.")

5FW7.pdb already exists, skipping download.
8W1N.pdb already exists, skipping download.
5DEJ.pdb already exists, skipping download.
3TFB.pdb already exists, skipping download.
3A4E.pdb already exists, skipping download.
3A4F.pdb already exists, skipping download.
6KGB.pdb already exists, skipping download.
5FW8.pdb already exists, skipping download.
2NOY.pdb already exists, skipping download.
4TKW.pdb already exists, skipping download.
3DJZ.pdb already exists, skipping download.
3DJR.pdb already exists, skipping download.
1X7T.pdb already exists, skipping download.
5CLY.pdb already exists, skipping download.
5CLZ.pdb already exists, skipping download.
1TTR.pdb already exists, skipping download.
8JNU.pdb already exists, skipping download.
8JOK.pdb already exists, skipping download.
4TL4.pdb already exists, skipping download.
3W3B.pdb already exists, skipping download.
1III.pdb already exists, skipping download.
3CXF.pdb already exists, skipping download.


In [12]:
parent_dir = os.path.dirname(cwd)
parent_dir

'/home/ugo/Scrivania/Transthyretin'

In [9]:
from Bio.PDB import Superimposer

def compute_rmsd(pdb1, pdb2, visualize=False):
    """
    Compute RMSD between two PDB files, handling missing residues.
    
    Parameters:
    -----------
    pdb1 : str
        Path to first PDB file (experimental structure)
    pdb2 : str
        Path to second PDB file (AlphaFold structure)
    visualize : bool
        Whether to return visualization widget
    
    Returns:
    --------
    float or tuple
        RMSD value, or (RMSD, visualization) if visualize=True
    """
    from Bio.PDB import PDBParser, Superimposer
    
    parser = PDBParser(QUIET=True)
    structure1 = parser.get_structure('pdb1', pdb1)
    structure2 = parser.get_structure('pdb2', pdb2)
    
    chains1 = get_chains(structure1)
    chains2 = get_chains(structure2)
        
    # Find common chains
    chain_ids1 = set(chain.id for chain in chains1)
    chain_ids2 = set(chain.id for chain in chains2)
    common_chains = chain_ids1.intersection(chain_ids2)
    
    if not common_chains:
        print(f"Chains in {pdb1}: {[chain.id for chain in chains1]}")
        print(f"Chains in {pdb2}: {[chain.id for chain in chains2]}")
        raise ValueError("No common chains found between structures") 
    
    min_rms = float('inf')
    min_chain1 = None
    min_chain2 = None
    min_atoms1 = None
    min_atoms2 = None

    for chain_id in chain_ids1:
        ca1_dict = {}
        chain1 = structure1[0][chain_id]
        for residue in chain1:
            if residue.has_id('CA'):
                ca1_dict[residue.id[1]] = residue['CA']
        
        for chain_id2 in chain_ids2:
            atoms2 = []
            atoms1 = []
            ca2_dict = {}
            chain2 = structure2[0][chain_id2]        
            for residue in chain2:
                if residue.has_id('CA'):
                    ca2_dict[residue.id[1]] = residue['CA']
        
            # Find common residue numbers
            common_residues = set(ca1_dict.keys()).intersection(set(ca2_dict.keys()))
            
            if not common_residues:
                print(f"Warning: No common residues found in chain {chain_id}")
                continue
            
            # Sort residues by number for consistent ordering
            common_residues = sorted(common_residues)
            
            #print(f"Chain {chain_id}: {len(common_residues)} common residues out of "
            #    f"{len(ca1_dict)} (structure1) and {len(ca2_dict)} (structure2)")
            
            # Add atoms in order
            for res_num in common_residues:
                atoms1.append(ca1_dict[res_num])
                atoms2.append(ca2_dict[res_num])
        
            if len(atoms1) != len(atoms2):
                raise ValueError(f"Structures have different number of CA atoms. {len(atoms1)} vs {len(atoms2)}")
        
            if len(atoms1) == 0:
                raise ValueError("No common CA atoms found between structures")
        
            # Perform superimposition
            super_imposer = Superimposer()
            super_imposer.set_atoms(atoms1, atoms2)
            rmsd = super_imposer.rms
            if rmsd < min_rms:
                min_rms = rmsd
                min_chain1 = chain_id
                min_chain2 = chain_id2
                min_atoms1 = atoms1
                min_atoms2 = atoms2
    
    print("Chain 1:", min_chain1)
    print("Chain 2:", min_chain2)
    # Apply transformation to structure2 for visualization
    if visualize:
        super_imposer.apply(min_atoms2)
        view = create_visualization(structure1, structure2, min_rms)
        return min_rms, view
    
    return min_rms

def get_chains(structure):
    """Get all chains from a structure."""
    return list(structure[0])

def get_missing_residues(structure):
    """
    Identify missing residues by looking for gaps in residue numbering.
    Returns a dictionary of missing residue numbers for each chain.
    """
    missing_residues = {}
    
    for chain in structure[0]:
        residue_numbers = []
        for residue in chain:
            if residue.id[0] == ' ':  # Only consider standard residues
                residue_numbers.append(residue.id[1])
        
        if residue_numbers:
            residue_numbers.sort()
            missing = []
            for i in range(min(residue_numbers), max(residue_numbers) + 1):
                if i not in residue_numbers:
                    missing.append(i)
            missing_residues[chain.id] = missing
    
    return missing_residues

def create_visualization(structure1, structure2, rmsd_value):
    """
    Create an NGLView visualization of superimposed structures.
    
    Parameters:
    -----------
    structure1 : Bio.PDB.Structure
        First structure (experimental)
    structure2 : Bio.PDB.Structure
        Second structure (AlphaFold, already superimposed)
    rmsd_value : float
        RMSD value between structures
    
    Returns:
    --------
    nglview.NGLWidget
        Visualization widget
    """
    try:
        import nglview as nv
    except ImportError:
        print("NGLView not installed. Install with: pip install nglview")
        return None
    
    # Create the view with both structures
    view = nv.show_biopython(structure1)
    view.add_component(nv.BiopythonStructure(structure2))
    
    # Clear default representations
    view.clear_representations()
    
    # Add cartoon representations
    view.add_representation('cartoon', selection='all', color='blue', 
                          component=0, opacity=0.8)
    view.add_representation('cartoon', selection='all', color='red', 
                          component=1, opacity=0.8)
    
    # Add backbone representation for better visibility
    view.add_representation('backbone', selection='all', color='darkblue', 
                          component=0, opacity=0.6)
    view.add_representation('backbone', selection='all', color='darkred', 
                          component=1, opacity=0.6)
    
    # Set camera and stage
    view.camera = 'perspective'
    view.stage.set_parameters(backgroundColor='white')
    
    print(f"\nVisualization created:")
    print(f"RMSD: {rmsd_value:.3f} Å")
    print("Blue: Experimental structure")
    print("Red: AlphaFold predicted structure")
    
    return view

def create_advanced_visualization(structure1, structure2, rmsd_value):
    """
    Create an advanced visualization with multiple representations.
    """
    try:
        import nglview as nv
    except ImportError:
        print("NGLView not installed. Install with: pip install nglview")
        return None
    
    view = nv.show_biopython(structure1)
    view.add_component(nv.BiopythonStructure(structure2))
    
    view.clear_representations()
    
    # Cartoon representations
    view.add_representation('cartoon', selection='all', color='blue', 
                          component=0, opacity=0.7)
    view.add_representation('cartoon', selection='all', color='red', 
                          component=1, opacity=0.7)
    
    # Surface representations for shape comparison
    view.add_representation('surface', selection='all', color='lightblue', 
                          component=0, opacity=0.3)
    view.add_representation('surface', selection='all', color='pink', 
                          component=1, opacity=0.3)
    
    # Highlight CA atoms
    view.add_representation('ball+stick', selection='@CA', color='navy', 
                          component=0, radius=0.3)
    view.add_representation('ball+stick', selection='@CA', color='maroon', 
                          component=1, radius=0.3)
    
    view.camera = 'perspective'
    view.stage.set_parameters(backgroundColor='white')
    
    print(f"\nAdvanced visualization created:")
    print(f"RMSD: {rmsd_value:.3f} Å")
    print("Blue/Navy: Experimental structure")
    print("Red/Maroon: AlphaFold predicted structure")
    print("Surfaces show overall shape differences")
    
    return view

def analyze_structures(pdb1, pdb2, advanced_viz=False):
    """
    Complete analysis function that computes RMSD and creates visualization.
    
    Parameters:
    -----------
    pdb1 : str
        Path to experimental PDB file
    pdb2 : str
        Path to AlphaFold PDB file
    advanced_viz : bool
        Whether to create advanced visualization
    
    Returns:
    --------
    tuple
        (RMSD value, visualization widget)
    """
    # Compute RMSD with visualization
    rmsd_value, view = compute_rmsd(pdb1, pdb2, visualize=True)
    
    # Create advanced visualization if requested
    if advanced_viz and view is not None:
        from Bio.PDB import PDBParser
        parser = PDBParser(QUIET=True)
        structure1 = parser.get_structure('pdb1', pdb1)
        structure2 = parser.get_structure('pdb2', pdb2)
        
        # Re-apply superimposition for advanced view
        atoms1, atoms2 = get_common_atoms(structure1, structure2)
        super_imposer = Superimposer()
        super_imposer.set_atoms(atoms1, atoms2)
        super_imposer.apply(structure2.get_atoms())
        
        view = create_advanced_visualization(structure1, structure2, rmsd_value)
    
    return rmsd_value, view

def get_common_atoms(structure1, structure2):
    """Helper function to get common CA atoms between structures."""
    atoms1 = []
    atoms2 = []
    
    chain_ids1 = set(chain.id for chain in structure1[0])
    chain_ids2 = set(chain.id for chain in structure2[0])
    common_chains = chain_ids1.intersection(chain_ids2)
    
    for chain_id in common_chains:
        chain1 = structure1[0][chain_id]
        chain2 = structure2[0][chain_id]
        
        ca1_dict = {}
        ca2_dict = {}
        
        for residue in chain1:
            if residue.has_id('CA'):
                ca1_dict[residue.id[1]] = residue['CA']
        
        for residue in chain2:
            if residue.has_id('CA'):
                ca2_dict[residue.id[1]] = residue['CA']
        
        common_residues = sorted(set(ca1_dict.keys()).intersection(set(ca2_dict.keys())))
        
        for res_num in common_residues:
            atoms1.append(ca1_dict[res_num])
            atoms2.append(ca2_dict[res_num])
    
    return atoms1, atoms2

In [14]:
alphafold_path = parent_dir + sep + "pdbs-alphafold" + sep + "tetramer"
rmsds = {}
views = {}

for index, row in df.iterrows():
    mutation = row['Mutation']
    pdb_id = row['PDB ID']
    if mutation != "WildType":
        original_aa = mutation[0].lower()
        mutated_aa = mutation[-1].lower()
        index = int(float(mutation[1:-1])) + 20
        mutation = f"{original_aa}{index}{mutated_aa}"
    else:
        mutation = "wt"

    print(f"Processing mutation: {mutation} ({pdb_id})")
    experimental_pdb = f"{path}{sep}{pdb_id}.pdb"
    alphafold_pdb = f"{alphafold_path}{sep}{mutation}-tetramer.pdb"
    rmsd, view = compute_rmsd(alphafold_pdb, experimental_pdb, visualize=True)
    rmsds[mutation] = rmsd
    views[mutation] = view
    print(f"RMSD for {mutation} ({pdb_id}): {rmsd:.2f} Å")

Processing mutation: a129v (5FW7)
Chain 1: B
Chain 2: B



Visualization created:
RMSD: 0.315 Å
Blue: Experimental structure
Red: AlphaFold predicted structure
RMSD for a129v (5FW7): 0.32 Å
Processing mutation: a140s (8W1N)
Chain 1: B
Chain 2: A

Visualization created:
RMSD: 0.225 Å
Blue: Experimental structure
Red: AlphaFold predicted structure
RMSD for a140s (8W1N): 0.23 Å
Processing mutation: a39d (5DEJ)
Chain 1: A
Chain 2: A

Visualization created:
RMSD: 15.286 Å
Blue: Experimental structure
Red: AlphaFold predicted structure
RMSD for a39d (5DEJ): 15.29 Å
Processing mutation: a45t (3TFB)
Chain 1: C
Chain 2: A

Visualization created:
RMSD: 0.270 Å
Blue: Experimental structure
Red: AlphaFold predicted structure
RMSD for a45t (3TFB): 0.27 Å
Processing mutation: e74g (3A4E)
Chain 1: C
Chain 2: A

Visualization created:
RMSD: 0.372 Å
Blue: Experimental structure
Red: AlphaFold predicted structure
RMSD for e74g (3A4E): 0.37 Å
Processing mutation: e74k (3A4F)
Chain 1: D
Chain 2: A

Visualization created:
RMSD: 0.238 Å
Blue: Experimental structur

In [15]:
rmsds 

{'a129v': 0.31544106831346663,
 'a140s': 0.2250830212210585,
 'a39d': 15.285905207253705,
 'a45t': 0.2697515324714926,
 'e74g': 0.37240081888644355,
 'e74k': 0.23793411884030205,
 'e81k': 0.19129574857854872,
 'e109k': 0.2715110197582054,
 'i104s': 0.26607189846897866,
 'l75p': 0.22582522512613487,
 'l78h': 0.24167070387116132,
 'r124h': 0.4222909626538759,
 's72p': 0.2122105789568266,
 't139m': 0.21978421015890026,
 'v142i': 0.25021742447789996,
 'v50g': 0.18251677913281036,
 'v50l': 0.17541643926858297,
 'v50m': 0.3080905067203694,
 'wt': 0.3565506488112287,
 'y134c': 0.27243616459783965,
 'y134h': 0.3071809136952519}

In [16]:
df

,Mutation,PDB ID
1,A109V,5FW7
2,A120S,8W1N
3,A19D,5DEJ
5,A25T,3TFB
28,E54G,3A4E
29,E54K,3A4F
33,E61K,6KGB
36,E89K,5FW8
71,I84S,2NOY
78,L55P,4TKW


In [17]:
problems_pdb = {mutation: rmsd for mutation, rmsd in rmsds.items() if rmsd > 2.0}
print("Mutations with RMSD > 2.0 Å:")
for mutation, rmsd in problems_pdb.items():
    print(f"{mutation}: {rmsd:.2f} Å")
problems_pdb = list(problems_pdb.keys())
problems_pdb

Mutations with RMSD > 2.0 Å:
a39d: 15.29 Å


['a39d']

In [18]:
views[problems_pdb[0]]

NGLWidget()

In [19]:
from Bio.PDB import PDBParser, PDBIO, Structure, Model, Chain
import numpy as np
from Bio.PDB.vectors import Vector, rotaxis2m

def dimer_to_tetramer(dimer_structure, symmetry_type="D2"):
    """
    Convert a dimer structure to a tetramer using biological symmetry operations.
    
    Parameters:
    - dimer_structure: Bio.PDB Structure object containing the dimer
    - symmetry_type: "D2" (dihedral), "C2" (cyclic), or "linear"
    
    Returns:
    - Bio.PDB Structure object containing the tetramer
    """
    
    # Create new structure for tetramer
    tetramer_structure = Structure.Structure("tetramer")
    tetramer_model = Model.Model(0)
    tetramer_structure.add(tetramer_model)
    
    # Get the original dimer chains
    original_chains = list(dimer_structure.get_chains())
    
    if symmetry_type == "D2":
        # Dihedral D2 symmetry - like the structure in your image
        # Two perpendicular 2-fold rotation axes
        transformations = [
            # Original dimer (identity)
            {"rotation": np.eye(3), "translation": np.array([0, 0, 0])},
            # 180° rotation around X-axis
            {"rotation": rotaxis2m(np.pi, Vector(1, 0, 0)), "translation": np.array([0, 0, 0])},
            # 180° rotation around Y-axis  
            {"rotation": rotaxis2m(np.pi, Vector(0, 1, 0)), "translation": np.array([0, 0, 0])},
            # 180° rotation around Z-axis (combination of X and Y rotations)
            {"rotation": rotaxis2m(np.pi, Vector(0, 0, 1)), "translation": np.array([0, 0, 0])}
        ]
    elif symmetry_type == "C2":
        # Cyclic C2 symmetry - simple 2-fold rotation
        transformations = [
            # Original dimer
            {"rotation": np.eye(3), "translation": np.array([0, 0, 0])},
            # 180° rotation around Z-axis
            {"rotation": rotaxis2m(np.pi, Vector(0, 0, 1)), "translation": np.array([0, 0, 0])}
        ]
    else:  # linear arrangement
        transformations = [
            # Original dimer
            {"rotation": np.eye(3), "translation": np.array([0, 0, 0])},
            # Translated dimer
            {"rotation": np.eye(3), "translation": np.array([50, 0, 0])}
        ]
    
    chain_counter = 0
    chain_ids = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']  # Extended for larger assemblies
    
    for i, transform in enumerate(transformations):
        for j, original_chain in enumerate(original_chains):
            # Create new chain with unique ID
            new_chain_id = chain_ids[chain_counter]
            new_chain = Chain.Chain(new_chain_id)
            
            # Copy and transform all residues
            for residue in original_chain:
                new_residue = residue.copy()
                
                # Transform all atoms in the residue
                for atom in new_residue:
                    # Apply rotation first, then translation
                    rotated_coord = np.dot(transform["rotation"], atom.coord)
                    new_coord = rotated_coord + transform["translation"]
                    atom.set_coord(new_coord)
                
                new_chain.add(new_residue)
            
            tetramer_model.add(new_chain)
            chain_counter += 1
            
            # Stop if we've created 4 chains for tetramer
            if chain_counter >= 4:
                break
        
        if chain_counter >= 4:
            break
    
    return tetramer_structure

def build_tetramer_from_dimer(pdb_file, output_file="tetramer.pdb", symmetry="D2"):
    """
    Complete pipeline to build tetramer from dimer PDB file.
    Mimics ProDy's buildBiomolecules functionality.
    """
    # Parse the dimer structure
    parser = PDBParser(QUIET=True)
    dimer_structure = parser.get_structure('dimer', pdb_file)
    
    # Create tetramer
    tetramer = dimer_to_tetramer(dimer_structure, symmetry)
    
    # Save the result
    io = PDBIO()
    io.set_structure(tetramer)
    io.save(output_file)
    
    print(f"Tetramer saved to {output_file}")
    print(f"Original dimer atoms: {len(list(dimer_structure.get_atoms()))}")
    print(f"Tetramer atoms: {len(list(tetramer.get_atoms()))}")
    
    return tetramer

def exclude_ions_and_build_tetramer(pdb_file, excluded_atoms=['K', 'NA', 'CL', 'MG'], symmetry="D2"):
    """
    Build tetramer while excluding specific atoms (like ions).
    Similar to ProDy's approach of excluding potassium ions.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('structure', pdb_file)
    
    # Create new structure without excluded atoms
    clean_structure = Structure.Structure("clean")
    clean_model = Model.Model(0)
    clean_structure.add(clean_model)
    
    for chain in structure.get_chains():
        new_chain = Chain.Chain(chain.id)
        for residue in chain:
            new_residue = residue.copy()
            # Remove excluded atoms
            atoms_to_remove = []
            for atom in new_residue:
                if atom.element in excluded_atoms:
                    atoms_to_remove.append(atom.id)
            
            for atom_id in atoms_to_remove:
                new_residue.detach_child(atom_id)
            
            if len(new_residue) > 0:  # Only add if residue has atoms left
                new_chain.add(new_residue)
        
        if len(new_chain) > 0:  # Only add if chain has residues left
            clean_model.add(new_chain)
    
    # Build tetramer from clean structure
    return dimer_to_tetramer(clean_structure, symmetry)


In [20]:
df

,Mutation,PDB ID
1,A109V,5FW7
2,A120S,8W1N
3,A19D,5DEJ
5,A25T,3TFB
28,E54G,3A4E
29,E54K,3A4F
33,E61K,6KGB
36,E89K,5FW8
71,I84S,2NOY
78,L55P,4TKW


In [21]:
# Using your existing parser setup
parser = PDBParser(QUIET=True)
dimer = parser.get_structure('dimer', path + sep + '5FW8.pdb')
tetramer = dimer_to_tetramer(dimer, "D2")
tetramer_af = parser.get_structure('tetramer_af', alphafold_path + sep + 'e109k-tetramer.pdb')

imposer = Superimposer()
atoms1 = []
atoms2 = []
for chain in tetramer.get_chains():
    for residue in chain:
        if residue.has_id('CA'):
            atoms1.append(residue['CA'])
for chain in tetramer_af.get_chains():
    for residue in chain:
        if residue.has_id('CA'):
            atoms2.append(residue['CA'])

print(f"Number of CA atoms in tetramer: {len(atoms1)}")
print(f"Number of CA atoms in tetramer AF: {len(atoms2)}")

if len(atoms1) != len(atoms2):
    common_atoms = get_common_atoms(tetramer, tetramer_af)

atoms1, atoms2 = common_atoms
imposer.set_atoms(atoms1, atoms2)
imposer.apply(tetramer_af.get_atoms())  
rmsd = imposer.rms
print(f"RMSD after superimposition: {rmsd:.2f}")

Number of CA atoms in tetramer: 464
Number of CA atoms in tetramer AF: 508
RMSD after superimposition: 52.70


In [22]:
def get_missing_residues_from_pdb(pdb_file):
    """Extract missing residues from REMARK 465 lines in PDB file."""
    missing_residues = {}
    
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith('REMARK 465'):
                # Skip header lines
                if 'RES C SSSEQI' in line or 'M RES C SSSEQI' in line:
                    continue

                # Parse the actual missing residue entries
                # Format: REMARK 465     RES_NAME CHAIN_ID RES_NUM
                parts = line.strip().split()
                if len(parts) == 5:
                    res_name = parts[2]
                    chain_id = parts[3] 
                    res_num = int(parts[4])
                    if chain_id not in missing_residues:
                        missing_residues[chain_id] = []
                    missing_residues[chain_id].append(res_num)
    
    # Sort the residue numbers for each chain
    for chain_id in missing_residues:
        missing_residues[chain_id] = sorted(missing_residues[chain_id])
    
    return missing_residues

# Usage
missing = get_missing_residues_from_pdb(path + sep + '5FW8.pdb')
print("Missing residues by chain:", missing)

Missing residues by chain: {'A': [1, 2, 3, 4, 5, 6, 7, 8, 9, 126, 127], 'B': [1, 2, 3, 4, 5, 6, 7, 8, 9, 126, 127]}


In [23]:
import numpy as np 
np.mean(list(rmsds.values())), np.std(list(rmsds.values()))

(0.9814088091077657, 3.1991828885705234)

In [24]:
rmsds = {k: v for k, v in rmsds.items() if v < 2.0}
print("RMSDs after filtering:", rmsds) 
np.mean(list(rmsds.values())), np.std(list(rmsds.values()))

RMSDs after filtering: {'a129v': 0.31544106831346663, 'a140s': 0.2250830212210585, 'a45t': 0.2697515324714926, 'e74g': 0.37240081888644355, 'e74k': 0.23793411884030205, 'e81k': 0.19129574857854872, 'e109k': 0.2715110197582054, 'i104s': 0.26607189846897866, 'l75p': 0.22582522512613487, 'l78h': 0.24167070387116132, 'r124h': 0.4222909626538759, 's72p': 0.2122105789568266, 't139m': 0.21978421015890026, 'v142i': 0.25021742447789996, 'v50g': 0.18251677913281036, 'v50l': 0.17541643926858297, 'v50m': 0.3080905067203694, 'wt': 0.3565506488112287, 'y134c': 0.27243616459783965, 'y134h': 0.3071809136952519}


(0.2661839892004689, 0.06350046463316397)

In [1]:
import os 
import pandas as pd

cwd = os.getcwd()
sep  = os.sep
parent_dir = os.path.dirname(cwd)

path = cwd + sep + "experimental_pdbs" + sep + "complex"
if not os.path.exists(path):
    os.makedirs(path)
csv_file = cwd + sep + "experimental_pdbs_complex.csv"
df = pd.read_csv(csv_file)
df

,mutation,tafamidis,acoramidis,diflunisal,tolcapone
0,WildType,6E6Z,NaN,6E70,NaN
1,A109S,NaN,NaN,NaN,NaN
2,A109V,NaN,NaN,NaN,NaN
3,A120S,NaN,NaN,NaN,NaN
4,A19D,NaN,NaN,NaN,NaN
...,...,...,...,...,...
127,Y114S,NaN,NaN,NaN,NaN
128,Y116S,NaN,NaN,NaN,NaN
129,Y69H,NaN,NaN,NaN,NaN
130,Y69I,NaN,NaN,NaN,NaN


In [2]:
ligands = ['tafamidis', 'diflunisal', 'tolcapone', 'acoramidis']
dict_structures = {}
for index, row in df.iterrows():
    mutation = row[0]
    for i, ligand in enumerate(ligands):
        pdb_id = row[i+1]
        if pd.isna(pdb_id) or pdb_id == 'NO':
            continue
        dict_structures[f"{mutation}_{ligand}"] = pdb_id
dict_structures

/tmp/ipykernel_40044/1132210072.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  mutation = row[0]
/tmp/ipykernel_40044/1132210072.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pdb_id = row[i+1]


{'WildType_tafamidis': '6E6Z',
 'WildType_tolcapone': '6E70',
 'A97S_tafamidis': '8YQD',
 'A97S_tolcapone': '7YCQ',
 'A97S_acoramidis': '7YBR',
 'D38A_tafamidis': '6E77',
 'D38A_tolcapone': '6E78',
 'L55P_tafamidis': '6E74',
 'S52P_tafamidis': '6FFT',
 'V122I_tafamidis': '4HIS',
 'V30M_tafamidis': '6E72',
 'V30M_tolcapone': '6E73'}

In [3]:
#Get pdbs from RCSB
for complex, pdb_id in dict_structures.items():
    if not os.path.exists(f"{path}{sep}{pdb_id}.pdb"):
        print(f"Downloading {pdb_id}.pdb")
        get_pdb(pdb_id)
        os.rename(f"{pdb_id}.pdb", f"{path}{sep}{pdb_id}.pdb")
    else:
        print(f"{pdb_id}.pdb already exists, skipping download.")

6E6Z.pdb already exists, skipping download.
6E70.pdb already exists, skipping download.
8YQD.pdb already exists, skipping download.
7YCQ.pdb already exists, skipping download.
7YBR.pdb already exists, skipping download.
6E77.pdb already exists, skipping download.
6E78.pdb already exists, skipping download.
6E74.pdb already exists, skipping download.
6FFT.pdb already exists, skipping download.
4HIS.pdb already exists, skipping download.
6E72.pdb already exists, skipping download.
6E73.pdb already exists, skipping download.


In [4]:
to_find = list(dict_structures.keys())
print(to_find)

docking_path = parent_dir + sep + "docking" + sep + "AutoDock" + sep + "docked-outputs" + sep + "existing"
dict_structures_docking = {}
for complex in to_find:
    mutation, ligand = complex.split('_')
    mutation = mutation.lower()
    if mutation == "wildtype":
        mutation = "wt"
    else:
        original_aa = mutation[0].lower()
        mutated_aa = mutation[-1].lower()
        index = int(float(mutation[1:-1])) + 20
        mutation = f"{original_aa}{index}{mutated_aa}"
    mutation = f"{mutation}-tetramer"
    filename = docking_path + sep + f"{mutation}_{ligand}.pdbqt"
    if not os.path.exists(filename):
        print(f"File {filename} does not exist, skipping.")
        continue
    dict_structures_docking[complex] = filename
dict_structures_docking

['WildType_tafamidis', 'WildType_tolcapone', 'A97S_tafamidis', 'A97S_tolcapone', 'A97S_acoramidis', 'D38A_tafamidis', 'D38A_tolcapone', 'L55P_tafamidis', 'S52P_tafamidis', 'V122I_tafamidis', 'V30M_tafamidis', 'V30M_tolcapone']


{'WildType_tafamidis': '/home/ugo/Scrivania/Transthyretin/docking/AutoDock/docked-outputs/existing/wt-tetramer_tafamidis.pdbqt',
 'WildType_tolcapone': '/home/ugo/Scrivania/Transthyretin/docking/AutoDock/docked-outputs/existing/wt-tetramer_tolcapone.pdbqt',
 'A97S_tafamidis': '/home/ugo/Scrivania/Transthyretin/docking/AutoDock/docked-outputs/existing/a117s-tetramer_tafamidis.pdbqt',
 'A97S_tolcapone': '/home/ugo/Scrivania/Transthyretin/docking/AutoDock/docked-outputs/existing/a117s-tetramer_tolcapone.pdbqt',
 'A97S_acoramidis': '/home/ugo/Scrivania/Transthyretin/docking/AutoDock/docked-outputs/existing/a117s-tetramer_acoramidis.pdbqt',
 'D38A_tafamidis': '/home/ugo/Scrivania/Transthyretin/docking/AutoDock/docked-outputs/existing/d58a-tetramer_tafamidis.pdbqt',
 'D38A_tolcapone': '/home/ugo/Scrivania/Transthyretin/docking/AutoDock/docked-outputs/existing/d58a-tetramer_tolcapone.pdbqt',
 'L55P_tafamidis': '/home/ugo/Scrivania/Transthyretin/docking/AutoDock/docked-outputs/existing/l75p-te

In [5]:
import pymol
from pymol import cmd

def Receptor3DView(receptorPDB, ligPDB, save_path):

    cmd.load(receptorPDB, "receptor")
    cmd.load(ligPDB, "ligand")
    cmd.save(save_path, "all")

In [6]:
mutants = list(dict_structures_docking.keys())
mutants = [mutant.split("_")[0].lower() for mutant in mutants]
mutants = list(set(mutants))
mutants

['a97s', 's52p', 'wildtype', 'd38a', 'l55p', 'v30m', 'v122i']

In [7]:
docked_structures_path = cwd + sep + "docked_pdbs"
pdb_dir = parent_dir + sep + "pdbs-alphafold" 
autodock_path = parent_dir + sep + "docking" + sep + "AutoDock"
autodock_ligand_path = autodock_path + sep + "docked-outputs" + sep + "existing"
dict_structures_docking = {}

for i, mutant in enumerate(mutants): 
    
    print(f"{i+1}/{len(mutants)}: Processing mutant {mutant}", end ="\r")

    if mutant == "wildtype":
        mutant = "wt-tetramer"
    else:
        original_aa = mutant[0].lower()
        mutated_aa = mutant[-1].lower()
        index = int(float(mutant[1:-1])) + 20
        mutant = f"{original_aa}{index}{mutated_aa}-tetramer"

    pdb_filepath = pdb_dir + sep + "tetramer" + sep + f"{mutant}.pdb"

    for ligand in ligands:
        output_path = docked_structures_path + sep + f"{mutant}_{ligand}_docked.pdb"
        ligandpdb =  autodock_ligand_path + sep + f"{mutant}_{ligand}.pdbqt"
        Receptor3DView(pdb_filepath, ligandpdb, output_path)
        dict_structures_docking[f"{mutant}_{ligand}"] = output_path

In [10]:
views = {}
rmsds = {}

if not os.path.exists(docked_structures_path):
    os.makedirs(docked_structures_path)

for complex, exp_structure in dict_structures.items():

    exp_structure = path + sep + f"{exp_structure}.pdb"
    
    mutant, ligand = complex.split('_')
    mutant = mutant.lower()
    if mutant == "wildtype":
        mutant = "wt-tetramer"
    else:
        original_aa = mutant[0].lower()
        mutated_aa = mutant[-1].lower()
        index = int(float(mutant[1:-1])) + 20
        mutant = f"{original_aa}{index}{mutated_aa}-tetramer"
    complex = f"{mutant}_{ligand}"

    if complex not in dict_structures_docking:
        print(f"Docked structure for {complex} not found, skipping.")
        continue
    docking_structure = dict_structures_docking[complex]
    print(f"Analyzing complex: {complex} (Experimental: {exp_structure}, Docked: {docking_structure})")

    output_path = docked_structures_path + sep + f"{complex}_docked.pdb"
        
    rmsd, view = compute_rmsd(docking_structure, exp_structure, visualize=True)
    print(f"RMSD for {complex}: {rmsd:.2f} Å")
    rmsds[complex] = rmsd
    # Store or display the view as needed
    #views[complex] = view

Analyzing complex: wt-tetramer_tafamidis (Experimental: /home/ugo/Scrivania/Transthyretin/experimental_validation/experimental_pdbs/complex/6E6Z.pdb, Docked: /home/ugo/Scrivania/Transthyretin/experimental_validation/docked_pdbs/wt-tetramer_tafamidis_docked.pdb)
Chain 1: B
Chain 2: A



Visualization created:
RMSD: 0.591 Å
Blue: Experimental structure
Red: AlphaFold predicted structure
RMSD for wt-tetramer_tafamidis: 0.59 Å
Analyzing complex: wt-tetramer_tolcapone (Experimental: /home/ugo/Scrivania/Transthyretin/experimental_validation/experimental_pdbs/complex/6E70.pdb, Docked: /home/ugo/Scrivania/Transthyretin/experimental_validation/docked_pdbs/wt-tetramer_tolcapone_docked.pdb)
Chain 1: B
Chain 2: A

Visualization created:
RMSD: 0.505 Å
Blue: Experimental structure
Red: AlphaFold predicted structure
RMSD for wt-tetramer_tolcapone: 0.51 Å
Analyzing complex: a117s-tetramer_tafamidis (Experimental: /home/ugo/Scrivania/Transthyretin/experimental_validation/experimental_pdbs/complex/8YQD.pdb, Docked: /home/ugo/Scrivania/Transthyretin/experimental_validation/docked_pdbs/a117s-tetramer_tafamidis_docked.pdb)
Chain 1: B
Chain 2: B

Visualization created:
RMSD: 0.525 Å
Blue: Experimental structure
Red: AlphaFold predicted structure
RMSD for a117s-tetramer_tafamidis: 0.53 Å


In [18]:
len(rmsds)

12

In [17]:
import numpy as np
mean_rmsd = np.mean(list(rmsds.values()))
std_rmsd = np.std(list(rmsds.values()))
print(f"Mean RMSD: {mean_rmsd:.4f} Å, Std Dev: {std_rmsd:.2f} Å")

Mean RMSD: 0.5083 Å, Std Dev: 0.11 Å


In [12]:
only_ligands_path = cwd + sep + "only_ligands"
if not os.path.exists(only_ligands_path):
    os.makedirs(only_ligands_path)

only_ligands_experimental_path = only_ligands_path + sep + "experimental"
if not os.path.exists(only_ligands_experimental_path):
    os.makedirs(only_ligands_experimental_path)

only_ligands_docked_path = only_ligands_path + sep + "docked"
if not os.path.exists(only_ligands_docked_path):
    os.makedirs(only_ligands_docked_path)

In [13]:
def extract_ligand_from_pdb(pdb_file, ligand_name, save_path):
    """
    Extract a specific ligand from a PDB file and save it as a separate PDB file.
    
    Parameters:
    - pdb_file: Path to the input PDB file
    - ligand_name: Name of the ligand to extract (e.g., '3MI' for Tafamidis)
    
    Returns:
    - Path to the saved ligand PDB file
    """
    from Bio.PDB import PDBParser, PDBIO
    from Bio.PDB import Structure, Model

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('structure', pdb_file)
    
    # Create a new structure for the ligand
    ligand_structure = Structure.Structure("ligand")
    ligand_model = Model.Model(0)
    ligand_structure.add(ligand_model)
    
    for chain in structure.get_chains():
        for residue in chain:
            if residue.resname == ligand_name:
                new_residue = residue.copy()
                ligand_model.add(new_residue)

    
    io = PDBIO()
    io.set_structure(ligand_structure)
    io.save(save_path)

In [14]:
dict_structures

{'WildType_tafamidis': '6E6Z',
 'WildType_tolcapone': '6E70',
 'A97S_tafamidis': '8YQD',
 'A97S_tolcapone': '7YCQ',
 'A97S_acoramidis': '7YBR',
 'D38A_tafamidis': '6E77',
 'D38A_tolcapone': '6E78',
 'L55P_tafamidis': '6E74',
 'S52P_tafamidis': '6FFT',
 'V122I_tafamidis': '4HIS',
 'V30M_tafamidis': '6E72',
 'V30M_tolcapone': '6E73'}

In [15]:
ligands_experimental_dict = {}
ligands_docked_dict = {}
ligand_names_exp = {
    "tafamidis": "3MI",
    "diflunisal": "1FL",
    "tolcapone": "TCW",
    "acoramidis": "16V"
}
ligand_names_dock = {
    "tafamidis": "UNL",
    "diflunisal": "UNL",
    "tolcapone": "UNL",
    "acoramidis": "UNL"
}

for complex, pdb_file in dict_structures_docking.items():
 
    mutation, ligand = complex.split('_')
    
    # Extract ligand from experimental structure
    exp_pdb_file = pdb_file
    save_path_exp = only_ligands_experimental_path + sep + f"{mutation}_{ligand}_experimental.pdb"
    extract_ligand_from_pdb(exp_pdb_file, ligand_names_exp[ligand], save_path_exp)
    ligands_experimental_dict[f"{mutation}_{ligand}"] = save_path_exp

    # Extract ligand from docked structure
    docked_pdb_file = pdb_file
    save_path_docked = only_ligands_docked_path + sep + f"{mutation}_{ligand}_docked.pdb"
    extract_ligand_from_pdb(docked_pdb_file, ligand_names_exp[ligand], save_path_docked)
    ligands_docked_dict[f"{mutation}_{ligand}"] = save_path_docked


In [16]:
#compute rmsd between the autodock ligand pose and the experimental one
rmsds_ligands = {}

for complex, pdb_file in ligands_experimental_dict.items():
    mutation, ligand = complex.split('_')
    ligand = ligand.upper()
    
    exp_ligand_pdb = pdb_file
    docked_ligand_pdb = ligands_docked_dict[complex]
    
    print(f"Analyzing ligand: {complex} (Experimental: {exp_ligand_pdb}, Docked: {docked_ligand_pdb})")
    
    rmsd, view = compute_rmsd(docked_ligand_pdb, exp_ligand_pdb, visualize=True)
    print(f"RMSD for {complex}: {rmsd:.2f} Å")
    rmsds_ligands[complex] = rmsd

Analyzing ligand: a117s-tetramer_tafamidis (Experimental: /home/ugo/Scrivania/Transthyretin/experimental_validation/only_ligands/experimental/a117s-tetramer_tafamidis_experimental.pdb, Docked: /home/ugo/Scrivania/Transthyretin/experimental_validation/only_ligands/docked/a117s-tetramer_tafamidis_docked.pdb)


KeyError: 0